# Desvendando Árvores de Decisão em Aprendizado de Máquina

### Resumo
Este notebook apresenta um guia prático para construir, treinar, visualizar e avaliar um modelo de classificação baseado no algoritmo de **Árvores de Decisão (Decision Tree)** utilizando a biblioteca Scikit-Learn. Usando um conjunto de dados simulado sobre seguros, aprenderemos a mapear características de motoristas e veículos para prever o nível de gravidade de acidentes.

---

### Objetivos de Aprendizagem
Ao final deste estudo, você será capaz de:
1. **Compreender o fluxo de preparação de dados**, desde a carga inicial até o tratamento de variáveis categóricas usando codificação de etiquetas (*Label Encoding*).
2. **Treinar e configurar um modelo de Árvore de Decisão** controlando hiperparâmetros essenciais como profundidade máxima e número de nós folha.
3. **Interpretar a estrutura visual da árvore** gerada graficamente para entender como o algoritmo toma suas decisões.
4. **Avaliar criticamente o desempenho do modelo** através de métricas de classificação fundamentais como Acurácia, Precisão, Recall e F1-Score.

### Pré-requisitos e Bibliotecas Usadas
Para este projeto, usamos o ecossistema padrão de Ciência de Dados em Python. Abaixo está o porquê de cada uma das bibliotecas importadas a seguir:
*   **`pandas (pd)`**: Ferramenta essencial para manipulação e análise de dados em formato de tabelas (DataFrames).
*   **`numpy (np)`**: Biblioteca matemática utilizada para manipulação eficiente de arranjos multidimensionais (vetores e matrizes).
*   **`sklearn.model_selection.train_test_split`**: Função responsável por dividir nossos dados de forma aleatória e justa entre uma parte para treino e outra para teste.
*   **`sklearn.tree.DecisionTreeClassifier`**: O coração do nosso notebook; o algoritmo que cria a estrutura lógica de decisão em formato de árvore.
*   **`sklearn.preprocessing.LabelEncoder`**: Ferramenta de pré-processamento que traduz textos (variáveis categóricas) em números, já que modelos matemáticos não leem palavras diretamente.
*   **`sklearn.metrics`**: Conjunto de funções (como `accuracy_score`, `precision_score`, etc.) que usamos para medir matematicamente o quão bom nosso modelo está.
*   **`graphviz` & `export_graphviz`**: Bibliotecas que transformam a estrutura interna da árvore de decisão treinada em uma imagem gráfica nítida e fácil de interpretar.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.tree import export_graphviz
import graphviz

## Passo 1: Carregamento do Dataset

### O que este passo faz
Nesta etapa, utilizamos o pandas para ler o arquivo `insurance.csv` e carregá-lo na memória como um DataFrame chamado `base`. O parâmetro `keep_default_na=False` é configurado para evitar que o pandas converta automaticamente textos vazios ou específicos em valores nulos (NaN), mantendo a integridade exata dos dados textuais.

### Por que ele é necessário
Antes de qualquer análise ou modelagem, precisamos trazer os dados do disco rígido para o ambiente de execução do Python de forma organizada e estruturada.

In [ ]:
base = pd.read_csv("insurance.csv", keep_default_na=False)
base.head()

,Unnamed: 0,GoodStudent,Age,SocioEcon,RiskAversion,VehicleYear,ThisCarDam,RuggedAuto,Accident,MakeModel,...,HomeBase,AntiTheft,PropCost,OtherCarCost,OtherCar,MedCost,Cushioning,Airbag,ILiCost,DrivHist
0,1,False,Adult,Prole,Adventurous,Older,Moderate,EggShell,Mild,Economy,...,City,False,TenThou,Thousand,True,Thousand,Poor,False,Thousand,Many
1,2,False,Senior,Prole,Cautious,Current,None,Football,None,Economy,...,City,True,Thousand,Thousand,True,Thousand,Good,True,Thousand,Zero
2,3,False,Senior,UpperMiddle,Psychopath,Current,None,Football,None,FamilySedan,...,City,False,Thousand,Thousand,False,Thousand,Good,True,Thousand,One
3,4,False,Adolescent,Middle,Normal,Older,None,EggShell,None,Economy,...,Suburb,False,Thousand,Thousand,True,Thousand,Fair,False,Thousand,Zero
4,5,False,Adolescent,Prole,Normal,Older,Moderate,Football,Moderate,Economy,...,City,False,TenThou,Thousand,False,Thousand,Fair,False,Thousand,Many


## Passo 2: Limpeza de Colunas Irrelevantes

### O que este passo faz
Removemos a coluna `Unnamed: 0` do nosso DataFrame utilizando a função `drop`. Essa coluna continha apenas índices numéricos herdados de exportações anteriores do arquivo.

### Por que ele é necessário
Manter colunas que servem apenas como identificadores ou índices no treinamento de modelos pode gerar ruído e confundir o algoritmo, pois elas não carregam nenhum padrão real sobre o fenômeno que queremos prever.

💡 **Para pensar:** Repare que na última linha da visualização do dataset (`base.head()`), a coluna `Unnamed: 0` desapareceu com sucesso, restando apenas os dados reais das propriedades.

In [ ]:
base = base.drop(columns=['Unnamed: 0'])
base.head()

,GoodStudent,Age,SocioEcon,RiskAversion,VehicleYear,ThisCarDam,RuggedAuto,Accident,MakeModel,DrivQuality,...,HomeBase,AntiTheft,PropCost,OtherCarCost,OtherCar,MedCost,Cushioning,Airbag,ILiCost,DrivHist
0,False,Adult,Prole,Adventurous,Older,Moderate,EggShell,Mild,Economy,Poor,...,City,False,TenThou,Thousand,True,Thousand,Poor,False,Thousand,Many
1,False,Senior,Prole,Cautious,Current,None,Football,None,Economy,Normal,...,City,True,Thousand,Thousand,True,Thousand,Good,True,Thousand,Zero
2,False,Senior,UpperMiddle,Psychopath,Current,None,Football,None,FamilySedan,Excellent,...,City,False,Thousand,Thousand,False,Thousand,Good,True,Thousand,One
3,False,Adolescent,Middle,Normal,Older,None,EggShell,None,Economy,Normal,...,Suburb,False,Thousand,Thousand,True,Thousand,Fair,False,Thousand,Zero
4,False,Adolescent,Prole,Normal,Older,Moderate,Football,Moderate,Economy,Poor,...,City,False,TenThou,Thousand,False,Thousand,Fair,False,Thousand,Many


## Passo 3: Divisão entre Variáveis Alvo (y) e Recursos (X)

### O que este passo faz
Dividimos nosso DataFrame em duas partes:
1. **`y`**: O vetor alvo (target) que contém o que queremos prever. No caso, extraímos a coluna de índice `7` (que corresponde à variável `Accident`, indicando a gravidade do acidente).
2. **`X`**: A matriz de características (features), contendo todos os dados preditores possíveis de um cliente, excluindo a própria coluna de acidentes.

### Por que ele é necessário
Para ensinar uma inteligência artificial, precisamos dar a ela as perguntas (as características do motorista/carro, `X`) separadas das respostas corretas (se houve acidente e qual a gravidade, `y`), de modo que ela aprenda a relação causal entre os dois lados.

In [ ]:
y = base.iloc[:,7].values
X  = base.drop(base.columns[7], axis=1).values
X

array([[False, 'Adult', 'Prole', ..., 'False', 'Thousand', 'Many'],
       [False, 'Senior', 'Prole', ..., 'True', 'Thousand', 'Zero'],
       [False, 'Senior', 'UpperMiddle', ..., 'True', 'Thousand', 'One'],
       ...,
       [False, 'Senior', 'Middle', ..., 'False', 'Thousand', 'Many'],
       [False, 'Adult', 'Prole', ..., 'False', 'Thousand', 'Zero'],
       [False, 'Adult', 'Middle', ..., '', '', '']],
      shape=(16280, 26), dtype=object)

## Passo 4: Codificação de Variáveis Categóricas (*Label Encoding*)

### O que este passo faz
Este trecho de código varre cada coluna da nossa matriz `X`. Se os dados contidos nela forem de texto (`dtype == 'object'`), usamos o `LabelEncoder` para converter esses textos em categorias numéricas sequenciais (por exemplo, transformando 'Adult', 'Senior', 'Adolescent' em `1, 2, 0`).

### Por que ele é necessário
Algoritmos de aprendizado de máquina realizam cálculos matemáticos de álgebra linear e probabilidade por baixo do capô. Eles não conseguem processar palavras diretamente; tudo precisa ser transformado em representações numéricas.

💡 **Para pensar:** Embora o *Label Encoding* seja prático para árvores de decisão, se usássemos regressões ou redes neurais, essa codificação numérica direta poderia implicar erroneamente uma ordem de importância matemática inexistente entre as categorias (ex: inferir que 'Senior' (2) é "maior" ou "melhor" que 'Adult' (1)). Para esses outros modelos, técnicas como *One-Hot Encoding* costumam ser preferidas.

In [ ]:
labelencoder = LabelEncoder()
for i in range(X.shape[1]):
  if X[:,i].dtype == 'object':
    X[:,i] = labelencoder.fit_transform(X[:,i])

## Passo 5: Divisão em Treino e Teste

### O que este passo faz
Separamos nossa base de dados em duas partes: $70\%$ para treinamento (`X_treinamento` e `y_treinamento`) e $30\%$ para teste (`X_teste` e `y_teste`). Definimos um `random_state=12` para garantir que essa separação seja reprodutível (ou seja, se rodarmos o código novamente, a divisão dos dados será exatamente a mesma).

### Por que ele é necessário
Se avaliarmos o modelo usando os mesmos dados que ele usou para aprender, não teremos como saber se ele realmente aprendeu a generalizar ou se apenas decorou as respostas (*overfitting*). A partição de teste atua como uma "prova de exame final" com perguntas inéditas para avaliar o modelo de verdade.

In [ ]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(X,y,test_size=0.3, random_state=12)

## Passo 6: Treinamento da Árvore de Decisão

### O que este passo faz
Instanciamos o nosso classificador `DecisionTreeClassifier` e o ajustamos aos nossos dados de treinamento por meio da função `fit`.
*   `max_depth=8`: Limita a profundidade máxima da árvore a 8 níveis.
*   `max_leaf_nodes=6`: Restringe a árvore a ter no máximo 6 nós finais (folhas) de decisão.

### Por que ele é necessário
É aqui que o aprendizado ocorre. O algoritmo analisa os dados de treino e descobre de forma autônoma quais perguntas dão o melhor poder de separação sobre a gravidade dos acidentes.

💡 **Para pensar:** Limitar a profundidade da árvore (`max_depth`) e o número de folhas (`max_leaf_nodes`) é uma técnica excelente de **poda precoce**. Sem essas travas, a árvore cresceria indefinidamente até decorar perfeitamente os dados de treino, o que causaria um desempenho péssimo ao enfrentar dados novos do mundo real.

In [ ]:
modelo = DecisionTreeClassifier(random_state=1,max_depth=8, max_leaf_nodes=6)
modelo.fit(X_treinamento,y_treinamento)

DecisionTreeClassifier(max_depth=8, max_leaf_nodes=6, random_state=1)

## Passo 7: Visualização Gráfica da Árvore

### O que este passo faz
Exportamos a estrutura interna do nosso modelo treinado no formato DOT e usamos a biblioteca `graphviz` para desenhar e renderizar o fluxo gráfico de tomada de decisão, salvando-o como uma imagem PNG.

### Por que ele é necessário
As Árvores de Decisão são modelos conhecidos como "caixas brancas", pois são fáceis de interpretar e explicáveis para seres humanos. Ao visualizar a árvore, conseguimos rastrear exatamente as regras lógicas que o modelo criou para classificar cada caso.

### Como interpretar o resultado
Cada caixa (nó) mostra a pergunta lógica feita sobre uma coluna (ex: `SeniorTrain <= 3.5`), a impureza de Gini (que mede quão misturadas estão as classes naquele ponto), o número de amostras que passaram por ali e a classe majoritária resultante.

In [ ]:
dot_data = export_graphviz(modelo, out_file=None, filled=True, feature_names=base.columns[:-1],
                           class_names=True, rounded=True)
graph = graphviz.Source(dot_data)
graph.render("decision_tree",format="png")
graph.view()

'decision_tree.pdf'

## Passo 8: Geração de Previsões

### O que este passo faz
Utilizamos a árvore já treinada para prever a gravidade de acidentes para os dados contidos em `X_teste` (dados que o modelo nunca viu antes durante o treinamento), armazenando as respostas inferidas na variável `previsoes`.

### Por que ele é necessário
Este passo simula o modelo operando no mundo real, gerando estimativas para novos cenários apresentados a ele.

In [ ]:
previsoes = modelo.predict(X_teste)

In [ ]:
previsoes

array(['None', 'None', 'None', ..., 'None', 'None', 'None'],
      shape=(4884,), dtype=object)

## Passo 9: Avaliação de Métricas de Desempenho

### O que este passo faz
Comparamos as previsões geradas pela nossa Árvore de Decisão com as respostas reais conhecidas (`y_teste`). Calculamos as quatro principais métricas de classificação:
*   **Acurácia**: Proporção geral de previsões corretas de todas as classes.
*   **Precisão**: De todas as vezes que o modelo previu uma classe, quantas vezes ele acertou de fato.
*   **Recall (Sensibilidade)**: De todas as ocorrências reais de uma determinada classe, quantas o modelo conseguiu capturar.
*   **F1-Score**: Uma média harmônica equilibrada entre Precisão e Recall.

### Como interpretar o resultado
O modelo obteve excelentes resultados, alcançando aproximadamente **$94,2\%$ de Acurácia** geral e métricas de Precisão e Recall muito equilibradas na casa dos $94\%$, mostrando que a árvore simplificada (com limite de 6 folhas) obteve um desempenho fantástico e robusto.

In [ ]:
accuracy = accuracy_score(y_teste,previsoes)
precision = precision_score(y_teste, previsoes, average='weighted')
recall = recall_score(y_teste, previsoes, average='weighted')
f1 = f1_score(y_teste, previsoes, average='weighted')
print(f'Acurácia: {accuracy}, Precisão: {precision}, Recall: {recall}, F1: {f1}')

Acurácia: 0.9426699426699426, Precisão: 0.9402207079954227, Recall: 0.9426699426699426, F1: 0.9402459769028098


## Passo 10: Relatório de Classificação Detalhado

### O que este passo faz
Gera um relatório detalhado classe por classe (`Mild`, `Moderate`, `None`, `Severe`) sobre o desempenho do modelo.

### Como interpretar o resultado
*   A classe **`None`** (sem acidentes) obteve o melhor desempenho absoluto com F1-score de `0.99`. Isso ocorre porque ela possui o maior volume de dados (`support = 3524`), facilitando o aprendizado.
*   A classe **`Moderate`** obteve o menor desempenho, com F1-score de `0.72`. Isso indica que a diferenciação entre acidentes moderados e leves/graves é uma tarefa mais sutil para as regras simples estabelecidas por essa árvore curta.

In [ ]:
report = classification_report(y_teste, previsoes)
print(report)

              precision    recall  f1-score   support

        Mild       0.87      0.73      0.80       413
    Moderate       0.79      0.67      0.72       394
        None       0.98      1.00      0.99      3524
      Severe       0.85      0.94      0.89       553

    accuracy                           0.94      4884
   macro avg       0.87      0.83      0.85      4884
weighted avg       0.94      0.94      0.94      4884



---

## Erros Comuns e Dicas de Resolução

*   **Erro ao Carregar o Dataset (`FileNotFoundError`)**: Geralmente ocorre se você reiniciar o ambiente do Colab e esquecer de fazer o upload do arquivo `insurance.csv` novamente. Lembre-se de sempre garantir que o arquivo esteja na barra lateral esquerda de arquivos.
*   **A árvore gerada está ilegível ou muito pequena**: Se você remover os limites `max_depth` e `max_leaf_nodes`, a árvore crescerá demais. O `graphviz` gerará uma imagem gigante onde os textos ficarão minúsculos e impossíveis de ler. Sempre controle o tamanho para fins didáticos!
*   **Erro de tipos não numéricos durante o treinamento**: Caso o loop de *Label Encoding* (Passo 4) pule alguma coluna por ela estar mal formatada, o algoritmo do Scikit-Learn falhará com um erro avisando que não consegue converter string para float.

---

## Resumo Final do Aprendizado

Neste laboratório prático aprendemos que:
*   **Dados brutos precisam de transformação**: Textos devem ser convertidos em representações numéricas sequenciais antes de serem inseridos em modelos de Machine Learning.
*   **As Árvores de Decisão são altamente interpretáveis**: Conseguimos ver o caminho lógico exato que o computador tomou para classificar um dado.
*   **A importância do controle de complexidade**: Controlar parâmetros como profundidade ajuda a manter o modelo simples, de fácil entendimento e evita que ele sofra de *overfitting* (decoreba).

---

## Exercícios Propostos

1.  **[FÁCIL] Explorando a complexidade:** Altere o hiperparâmetro `max_depth` do modelo para `5` e configure `max_leaf_nodes=10`. Treine novamente o classificador, exiba a árvore gerada e verifique se a Acurácia geral melhorou ou piorou.
2.  **[MÉDIO] Divisão de Treino alternativa:** Modifique a função `train_test_split` para separar $80\%$ dos dados para treinamento e $20\%$ para teste. Mude o valor de `random_state` para `42`. Avalie o impacto disso nas métricas de desempenho final do modelo.
3.  **[DESAFIADOR] Tratando classes desbalanceadas:** Como vimos no relatório de classificação, o dataset possui muitos registros da classe `None` (sem acidentes) e poucos das demais. Pesquise sobre o parâmetro `class_weight` dentro do `DecisionTreeClassifier` e configure-o como `class_weight='balanced'`. Treine o modelo e observe se o F1-score das categorias menores (`Mild`, `Moderate`, `Severe`) melhorou.

---

## Para ir além
Se você deseja aprofundar seus conhecimentos em Inteligência Artificial, pesquise sobre estes tópicos:
*   **Entropia vs. Impureza de Gini**: Quais são as diferenças matemáticas nos critérios de divisão que a árvore de decisão usa para cortar os dados?
*   **Random Forests (Florestas Aleatórias)**: Como combinar várias árvores de decisão individuais para criar um modelo ensemble incrivelmente mais forte e menos suscetível a erros?
*   **Overfitting e Underfitting**: Entenda o equilíbrio tênue que existe na modelagem preditiva entre simplificar demais um modelo versus torná-lo complexo demais.